<a href="https://colab.research.google.com/github/winzepz/Artificial-Intelligence-and-Machine-Learning/blob/main/W5W6CNN_Complete.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# WORKSHEET 5


## Imports

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten, Dense,
    BatchNormalization, Dropout, Activation,
    GlobalAveragePooling2D, Input
)
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os
import random
from PIL import Image, UnidentifiedImageError
from keras.utils import to_categorical
from sklearn.metrics import classification_report

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

## Section 1 — Simple CNN on MNIST (Keras Demo)

In [ ]:
# Load MNIST handwritten digits
(x_train_mnist, y_train_mnist), (x_test_mnist, y_test_mnist) = keras.datasets.mnist.load_data()
print("MNIST train shape:", x_train_mnist.shape)
print("MNIST test shape :", x_test_mnist.shape)

In [ ]:
# Normalise and add channel dimension
x_train_mnist = x_train_mnist.astype("float32") / 255.0
x_test_mnist  = x_test_mnist.astype("float32")  / 255.0
x_train_mnist = np.expand_dims(x_train_mnist, axis=-1)
x_test_mnist  = np.expand_dims(x_test_mnist,  axis=-1)
print("After reshape — train:", x_train_mnist.shape, "test:", x_test_mnist.shape)

In [ ]:
# Build simple CNN
mnist_model = keras.Sequential([
    layers.Conv2D(32, (3, 3), activation="relu", input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(10, activation="softmax")  # 10 digit classes
])
mnist_model.summary()

In [ ]:
mnist_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
mnist_model.fit(
    x_train_mnist, y_train_mnist,
    epochs=5,
    batch_size=32,
    validation_data=(x_test_mnist, y_test_mnist)
)

In [ ]:
test_loss, test_acc = mnist_model.evaluate(x_test_mnist, y_test_mnist)
print(f"MNIST Test Accuracy: {test_acc:.4f}")

In [ ]:
# Predictions on first 5 test samples
predictions   = mnist_model.predict(x_test_mnist[:5])
pred_labels   = np.argmax(predictions, axis=1)
print("Predicted labels:", pred_labels)
print("Actual labels:   ", y_test_mnist[:5])

---
## Exercise — End-to-End CNN for FruitsInAmazon Dataset
### Task 1: Data Understanding and Visualization

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Dataset paths
dataset_path = "/content/drive/MyDrive/Colab Notebooks/AI and Machine Learning/Week5/FruitinAmazon/FruitinAmazon"
train_path   = os.path.join(dataset_path, "train")
test_path    = os.path.join(dataset_path, "test")
print("Contents of dataset folder:", os.listdir(dataset_path))

In [ ]:
# 1a. Get class names from subdirectories
class_names = sorted([
    d for d in os.listdir(train_path)
    if os.path.isdir(os.path.join(train_path, d))
])
NUM_CLASSES = len(class_names)
print(f"Found {NUM_CLASSES} classes: {class_names}")

In [ ]:
# 1b. Select one random image per class and display in a 2-row grid
selected_images = []
selected_labels = []

for class_name in class_names:
    class_path = os.path.join(train_path, class_name)
    images = [img for img in os.listdir(class_path)
               if img.lower().endswith(('.png', '.jpg', '.jpeg'))]
    if images:
        selected_images.append(os.path.join(class_path, random.choice(images)))
        selected_labels.append(class_name)

num_cls = len(selected_images)
cols    = (num_cls + 1) // 2
fig, axes = plt.subplots(2, cols, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    if i < num_cls:
        ax.imshow(mpimg.imread(selected_images[i]))
        ax.set_title(selected_labels[i])
        ax.axis("off")
    else:
        ax.axis("off")
plt.suptitle("One Random Image per Class", fontsize=14)
plt.tight_layout()
plt.show()

**Observation:** Each class folder contains fruit images with varying backgrounds, lighting conditions, and sizes. The dataset appears balanced with distinct fruit categories.

In [ ]:
# 1c. Check for corrupted images and remove them
corrupted = []
for class_name in class_names:
    class_path = os.path.join(train_path, class_name)
    if not os.path.exists(class_path):
        continue
    for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path, img_name)
        if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
            try:
                with Image.open(img_path) as img:
                    img.verify()
            except (IOError, SyntaxError, UnidentifiedImageError, OSError) as e:
                # Check if it's a connection error or an actual corrupt file
                if "Transport endpoint is not connected" in str(e):
                    print("Drive connection lost. Please remount Google Drive.")
                    break

                corrupted.append(img_path)
                if os.path.exists(img_path):
                    os.remove(img_path)
                    print(f"Removed corrupted image: {img_path}")

if not corrupted:
    print("No corrupted images found (or drive disconnected).")
else:
    print(f"Total removed: {len(corrupted)}")

### Task 2: Loading and Preprocessing Image Data in Keras

In [ ]:
IMG_HEIGHT = 128
IMG_WIDTH  = 128
BATCH_SIZE = 32
VAL_SPLIT  = 0.2

rescale = tf.keras.layers.Rescaling(1./255)

# Training dataset
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_path,
    labels='inferred',
    label_mode='int',
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    interpolation='nearest',
    batch_size=BATCH_SIZE,
    shuffle=True,
    validation_split=VAL_SPLIT,
    subset='training',
    seed=123
)

# Validation dataset
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_path,
    labels='inferred',
    label_mode='int',
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    interpolation='nearest',
    batch_size=BATCH_SIZE,
    shuffle=False,
    validation_split=VAL_SPLIT,
    subset='validation',
    seed=123
)

# Normalise
train_ds = train_ds.map(lambda x, y: (rescale(x), y))
val_ds   = val_ds.map(lambda x, y: (rescale(x), y))

print("Datasets loaded and normalised.")

### Task 3: Implement CNN (Week 5 Architecture)

In [ ]:
model_w5 = keras.Sequential([
    layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)),

    # Conv Block 1
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),

    # Conv Block 2
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),

    # Fully Connected
    layers.Flatten(),
    layers.Dense(64,  activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(NUM_CLASSES, activation='softmax')
])
model_w5.summary()

### Task 4: Compile & Train the Model

In [ ]:
model_w5.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
print("Week 5 model compiled.")

In [ ]:
ckpt_w5 = ModelCheckpoint(
    'best_model_w5.keras',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)
es_w5 = EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True,
    verbose=1
)

history_w5 = model_w5.fit(
    train_ds,
    epochs=250,
    batch_size=16,
    validation_data=val_ds,
    callbacks=[ckpt_w5, es_w5]
)

### Task 5: Evaluate the Model

In [ ]:
val_loss_w5, val_acc_w5 = model_w5.evaluate(val_ds)
print(f"Week 5 — Validation Loss    : {val_loss_w5:.4f}")
print(f"Week 5 — Validation Accuracy: {val_acc_w5:.4f}")

### Task 6: Save and Load the Model

In [ ]:
# Save as .h5
model_w5.save('fruit_cnn_week5.h5')
print("Saved: fruit_cnn_week5.h5")

# Load and re-evaluate
loaded_w5 = keras.models.load_model('fruit_cnn_week5.h5')
_, reload_acc = loaded_w5.evaluate(val_ds)
print(f"Reloaded model — Validation Accuracy: {reload_acc:.4f}")

### Task 7: Predictions & Classification Report

In [ ]:
# Collect all predictions from validation set
y_true_w5, y_pred_w5 = [], []
for images, labels in val_ds:
    preds = model_w5.predict(images, verbose=0)
    y_pred_w5.extend(np.argmax(preds, axis=1))
    y_true_w5.extend(labels.numpy())

print("--- Week 5: Fruit Classification Report ---")
print(classification_report(y_true_w5, y_pred_w5, target_names=class_names, labels=range(NUM_CLASSES)))

In [ ]:
# Training / Validation curves
h5 = history_w5.history

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(h5['accuracy'],     label='Train Accuracy', marker='o')
plt.plot(h5['val_accuracy'], label='Val Accuracy',   marker='o')
plt.title('Week 5 — Model Accuracy')
plt.xlabel('Epoch'); plt.ylabel('Accuracy')
plt.legend(); plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(h5['loss'],     label='Train Loss', color='red',    marker='o')
plt.plot(h5['val_loss'], label='Val Loss',   color='orange', marker='o')
plt.title('Week 5 — Model Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.legend(); plt.grid(True)

plt.tight_layout()
plt.show()

---
# ══════════════════════════════════════
# WORKSHEET 6
# ══════════════════════════════════════

## Section 2.1 — Data Understanding and Visualizations (Worksheet 6 Style)

In [ ]:
# Re-read class names (same train_path from above)
class_names = sorted(os.listdir(train_path))
print(f"Found {len(class_names)} classes: {class_names}")

In [ ]:
# Check for corrupted images
corrupted_images = []
for class_name in class_names:
    class_path = os.path.join(train_path, class_name)
    if os.path.isdir(class_path):
        for img_name in os.listdir(class_path):
            img_path = os.path.join(class_path, img_name)
            try:
                with Image.open(img_path) as img:
                    img.verify()
            except (IOError, UnidentifiedImageError, SyntaxError):
                corrupted_images.append(img_path)
                os.remove(img_path)
                print(f"Removed corrupted image: {img_path}")

print("No corrupted images found." if not corrupted_images
      else f"Total removed: {len(corrupted_images)}")

In [ ]:
# Class balance
class_counts = {}
for class_name in class_names:
    class_path = os.path.join(train_path, class_name)
    if os.path.isdir(class_path):
        imgs = [f for f in os.listdir(class_path)
                if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        class_counts[class_name] = len(imgs)

print("\nClass Distribution:")
print("=" * 45)
print(f"{'Class Name':<25}{'Valid Image Count':>15}")
print("=" * 45)
for name, count in class_counts.items():
    print(f"{name:<25}{count:>15}")
print("=" * 45)

In [ ]:
# Random image visualisation — 2-row grid
sel_imgs, sel_lbls = [], []
for class_name in class_names:
    class_path = os.path.join(train_path, class_name)
    if os.path.isdir(class_path):
        imgs = [f for f in os.listdir(class_path)
                if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        if imgs:
            sel_imgs.append(os.path.join(class_path, random.choice(imgs)))
            sel_lbls.append(class_name)

n    = len(sel_imgs)
cols = (n + 1) // 2
fig, axes = plt.subplots(2, cols, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    if i < n:
        ax.imshow(mpimg.imread(sel_imgs[i]))
        ax.set_title(sel_lbls[i])
        ax.axis("off")
    else:
        ax.axis("off")
plt.suptitle("Random Sample — Each Class", fontsize=14)
plt.tight_layout()
plt.show()

## Section 2.2 — Data Generation, Augmentation & Pre-processing

In [ ]:
# Load datasets using keras.utils.image_dataset_from_directory (new API)
IMAGE_SIZE  = (128, 128)
BATCH_SIZE  = 32
NUM_CLASSES = len(class_names)

train_ds_w6, val_ds_w6 = keras.utils.image_dataset_from_directory(
    train_path,
    validation_split=0.2,
    subset="both",
    seed=1337,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

for images, labels in train_ds_w6.take(1):
    print("Images batch shape:", images.shape)
    print("Labels batch shape:", labels.shape)

In [ ]:
# Visualise one raw training batch
plt.figure(figsize=(10, 10))
for images, labels in train_ds_w6.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(np.array(images[i]).astype("uint8"))
        plt.title(class_names[int(labels[i])])
        plt.axis("off")
plt.suptitle("Sample Training Images (Before Augmentation)", fontsize=13)
plt.tight_layout()
plt.show()

### Data Augmentation — using tf.keras.layers

In [ ]:
# Define augmentation layers — GPU-accelerated, only active during training
data_augmentation_layers = [
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(height_factor=0.1, width_factor=0.1),
    layers.RandomContrast(0.1),
]

def data_augmentation(images):
    for layer in data_augmentation_layers:
        images = layer(images)
    return images

print("Augmentation layers:", [l.name for l in data_augmentation_layers])

In [ ]:
# Visualise 9 augmented versions of the same image
plt.figure(figsize=(10, 10))
for images, _ in train_ds_w6.take(1):
    for i in range(9):
        aug = data_augmentation(images)
        ax  = plt.subplot(3, 3, i + 1)
        plt.imshow(np.array(aug[0]).astype("uint8"))
        plt.axis("off")
plt.suptitle("9 Augmented Versions of the Same Image", fontsize=13)
plt.tight_layout()
plt.show()

---
## Task 1 — Improved CNN with BatchNormalization & Dropout
**Improvements over Week 5:**
- Data Augmentation baked into the model (GPU-accelerated)
- Rescaling inside the model
- BatchNormalization after every Conv2D and Dense layer
- Dropout to prevent overfitting
- Deeper architecture: 4 convolutional blocks + 3 dense layers

In [ ]:
input_shape = (IMAGE_SIZE[0], IMAGE_SIZE[1], 3)

model_task1 = Sequential([
    Input(shape=input_shape),

    # --- Augmentation (only active during model.fit) ---
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(height_factor=0.1, width_factor=0.1),

    # --- Rescaling: [0,255] → [0,1] ---
    layers.Rescaling(1./255),

    # === Block 1: 32 filters ===
    Conv2D(32, (3, 3), padding='same', activation=None),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    # === Block 2: 64 filters ===
    Conv2D(64, (3, 3), padding='same', activation=None),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    # === Block 3: 128 filters ===
    Conv2D(128, (3, 3), padding='same', activation=None),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    # === Block 4: 256 filters ===
    Conv2D(256, (3, 3), padding='same', activation=None),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    Flatten(),

    # === FC Layer 1 ===
    Dense(512, activation=None),
    BatchNormalization(),
    Activation('relu'),
    Dropout(0.5),

    # === FC Layer 2 ===
    Dense(256, activation=None),
    BatchNormalization(),
    Activation('relu'),
    Dropout(0.5),

    # === FC Layer 3 ===
    Dense(128, activation=None),
    BatchNormalization(),
    Activation('relu'),
    Dropout(0.5),

    # === Output Layer ===
    Dense(NUM_CLASSES, activation='softmax')
])

model_task1.summary()

In [ ]:
model_task1.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
print("Task 1 model compiled.")

In [ ]:
ckpt_t1 = ModelCheckpoint(
    'best_model_task1.keras',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)
es_t1 = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

history_task1 = model_task1.fit(
    train_ds_w6,
    epochs=100,
    validation_data=val_ds_w6,
    callbacks=[ckpt_t1, es_t1]
)

In [ ]:
# Training curves
h1 = history_task1.history
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(h1['accuracy'],     label='Train Accuracy', marker='o')
plt.plot(h1['val_accuracy'], label='Val Accuracy',   marker='o')
plt.title('Task 1 — Improved CNN Accuracy')
plt.xlabel('Epoch'); plt.ylabel('Accuracy')
plt.legend(); plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(h1['loss'],     label='Train Loss', color='red',    marker='o')
plt.plot(h1['val_loss'], label='Val Loss',   color='orange', marker='o')
plt.title('Task 1 — Improved CNN Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.legend(); plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
val_loss_t1, val_acc_t1 = model_task1.evaluate(val_ds_w6)
print(f"Task 1 — Validation Loss    : {val_loss_t1:.4f}")
print(f"Task 1 — Validation Accuracy: {val_acc_t1:.4f}")

In [ ]:
# Save as .keras and .h5
model_task1.save('fruit_cnn_task1.keras')
model_task1.save('fruit_cnn_task1.h5')
print("Saved: fruit_cnn_task1.keras & fruit_cnn_task1.h5")

# Reload and verify
loaded_t1 = keras.models.load_model('fruit_cnn_task1.keras')
_, reload_acc_t1 = loaded_t1.evaluate(val_ds_w6)
print(f"Reloaded model — Validation Accuracy: {reload_acc_t1:.4f}")

In [ ]:
# Classification Report — Task 1
y_true_t1, y_pred_t1 = [], []
for images, labels in val_ds_w6:
    preds = model_task1.predict(images, verbose=0)
    y_pred_t1.extend(np.argmax(preds, axis=1))
    y_true_t1.extend(labels.numpy())

print("--- Task 1: Classification Report (Improved CNN with BN + Dropout) ---")
print(classification_report(y_true_t1, y_pred_t1, target_names=class_names))

### Observation — Task 1 vs Week 5
The improved model is expected to outperform the Week 5 baseline because:
- **Data Augmentation** artificially enlarges the training set, reducing overfitting.
- **BatchNormalization** stabilises training by normalising layer inputs, leading to faster convergence.
- **Dropout** forces the network to learn distributed, robust representations rather than memorising training data.
- A **deeper architecture** (4 conv blocks) captures more complex and abstract visual features.

---
## Task 2 — Transfer Learning with VGG16

In [ ]:
# VGG16 requires 224x224 RGB images
VGG_SIZE = (224, 224)

train_ds_vgg, val_ds_vgg = keras.utils.image_dataset_from_directory(
    train_path,
    validation_split=0.2,
    subset='both',
    seed=1337,
    image_size=VGG_SIZE,
    batch_size=BATCH_SIZE,
)

# VGG16 preprocess_input scales to [-1, 1] as expected by the model
train_ds_vgg = train_ds_vgg.map(lambda x, y: (preprocess_input(x), y))
val_ds_vgg   = val_ds_vgg.map(lambda x, y: (preprocess_input(x), y))

for imgs, lbls in train_ds_vgg.take(1):
    print("VGG train batch shape:", imgs.shape)

In [ ]:
# Step 1: Load pre-trained VGG16 (without top classification layers)
base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)
print(f"VGG16 loaded. Total layers: {len(base_model.layers)}")

In [ ]:
# Step 2: Freeze all base model layers
for layer in base_model.layers:
    layer.trainable = False

trainable = sum(1 for l in base_model.layers if l.trainable)
print(f"Trainable layers in base: {trainable} / {len(base_model.layers)}")

In [ ]:
# Step 3: Add custom classification head
x = base_model.output
x = GlobalAveragePooling2D()(x)                          # Reduce spatial dims
x = Dense(1024, activation='relu')(x)                   # FC layer
x = Dropout(0.5)(x)                                     # Regularisation
x = Dense(NUM_CLASSES, activation='softmax')(x)         # Output

# Step 4: Create final model
model_vgg = Model(inputs=base_model.input, outputs=x)
model_vgg.summary()

In [ ]:
# Step 5: Compile
model_vgg.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
print("VGG16 model compiled.")

In [ ]:
ckpt_vgg = ModelCheckpoint(
    'best_model_vgg16.keras',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)
es_vgg = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

history_vgg = model_vgg.fit(
    train_ds_vgg,
    epochs=50,
    validation_data=val_ds_vgg,
    callbacks=[ckpt_vgg, es_vgg]
)

In [ ]:
# Training curves — VGG16
hv = history_vgg.history
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(hv['accuracy'],     label='Train Accuracy', marker='o')
plt.plot(hv['val_accuracy'], label='Val Accuracy',   marker='o')
plt.title('VGG16 Transfer Learning — Accuracy')
plt.xlabel('Epoch'); plt.ylabel('Accuracy')
plt.legend(); plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(hv['loss'],     label='Train Loss', color='red',    marker='o')
plt.plot(hv['val_loss'], label='Val Loss',   color='orange', marker='o')
plt.title('VGG16 Transfer Learning — Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.legend(); plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
vgg_loss, vgg_acc = model_vgg.evaluate(val_ds_vgg)
print(f"VGG16 — Validation Loss    : {vgg_loss:.4f}")
print(f"VGG16 — Validation Accuracy: {vgg_acc:.4f}")

In [ ]:
# Classification Report — VGG16
y_true_vgg, y_pred_vgg = [], []
for images, labels in val_ds_vgg:
    preds = model_vgg.predict(images, verbose=0)
    y_pred_vgg.extend(np.argmax(preds, axis=1))
    y_true_vgg.extend(labels.numpy())

print("--- Task 2: Classification Report (VGG16 Transfer Learning) ---")
print(classification_report(y_true_vgg, y_pred_vgg, target_names=class_names))

In [ ]:
# Inference visualisation — colour-coded (green=correct, red=wrong)
plt.figure(figsize=(12, 12))
for images, labels in val_ds_vgg.take(1):
    preds = model_vgg.predict(images, verbose=0)
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        disp = images[i].numpy()
        disp = (disp - disp.min()) / (disp.max() - disp.min() + 1e-8)
        plt.imshow(disp)
        true_lbl = class_names[int(labels[i])]
        pred_lbl = class_names[np.argmax(preds[i])]
        conf     = np.max(preds[i]) * 100
        color    = 'green' if true_lbl == pred_lbl else 'red'
        plt.title(f"True : {true_lbl}\nPred : {pred_lbl} ({conf:.1f}%)",
                  color=color, fontsize=9)
        plt.axis('off')
plt.suptitle("VGG16 Inference — Green = Correct | Red = Wrong", fontsize=13)
plt.tight_layout()
plt.show()